# 1.2 Train preprocessing

Clean **train only**: drop duplicates and missing rows. Fit a preprocessor on train features (same columns, no new features). Save CSV and `.pkl`.

| Artifact | Path |
|---|---|
| Input | `1-experimentation/data/data_raw_train.csv` |
| Output CSV | `1-experimentation/data/data_train_preprocessed.csv` |
| Output preprocessor | `1-experimentation/models/preprocessor.pkl` |


In [1]:
%pip install -q pandas scikit-learn



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pickle
from pathlib import Path

import pandas as pd
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

set_config(transform_output="pandas")

TARGET = "price"
FEATURE_NUMERIC = ["sqft", "bedrooms", "bathrooms", "year_built"]
FEATURE_CATEGORICAL = ["location", "condition"]

TRAIN_PATH = "../data/data_raw_train.csv"
TRAIN_PREPROCESSED_PATH = "../data/data_train_preprocessed.csv"
PREPROCESSOR_PATH = "../models/preprocessor.pkl"

Path("../models").mkdir(parents=True, exist_ok=True)


## 1. Load and clean train

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
print(f"Before: {train_df.shape[0]} rows, duplicates={train_df.duplicated().sum()}, missing={int(train_df.isna().sum().sum())}")

train_clean = train_df.drop_duplicates().dropna().reset_index(drop=True)
print(f"After:  {train_clean.shape[0]} rows, duplicates={train_clean.duplicated().sum()}, missing={int(train_clean.isna().sum().sum())}")
train_clean.head()


Before: 68 rows, duplicates=0, missing=1
After:  67 rows, duplicates=0, missing=0


,price,sqft,bedrooms,bathrooms,location,year_built,condition
0,495000.0,1950,3,2.0,Urban,1981,Good
1,320000.0,1700,2,1.5,Rural,1961,Fair
2,615000.0,2230,3,2.0,Downtown,1986,Good
3,398000.0,1680,2,2.0,Suburb,1968,Fair
4,357000.0,1580,2,1.5,Suburb,1960,Fair


## 2. Fit preprocessor and save artifacts

Impute numeric columns with the median and categorical columns with the most frequent value. Fitted on train features only; `price` is not included.


In [ ]:
X_train = train_clean.drop(columns=[TARGET])
y_train = train_clean[TARGET]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), FEATURE_NUMERIC),
        ("cat", SimpleImputer(strategy="most_frequent"), FEATURE_CATEGORICAL),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preprocessor.fit(X_train)
train_preprocessed = preprocessor.transform(X_train)
train_preprocessed[TARGET] = y_train.to_numpy()
train_preprocessed = train_preprocessed[list(train_clean.columns)]

train_preprocessed.to_csv(TRAIN_PREPROCESSED_PATH, index=False)
with open(PREPROCESSOR_PATH, "wb") as file:
    pickle.dump(preprocessor, file)

print(f"Wrote {TRAIN_PREPROCESSED_PATH} ({len(train_preprocessed)} rows)")
print(f"Wrote {PREPROCESSOR_PATH}")
